# Chromosome, Replicate 1 Scripting

Create a script to run the deconvolution on each replicate. This notebook will test the script.


In [31]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
from pipeline.fit_replication_profiles import main
from src.utils import mkdir_safe


output_dir = "output/prototype_run"
mkdir_safe(output_dir)

parameter_updates_df, optimizer, replication_deconvolver = main(chrom=1,
    replicate=2, num_epochs=100, output_directory=output_dir)


Creating directory: output/prototype_run...Directory exists. Skipping.
Loading initial cell cycle parameters from CLOCCS fits.
todo: Loading testing config from replication deconvolution
Threshold windows with less than 75% read coverage.
Running initial iterations...
Initializing N using config H.
Initializing B using timepoint 0
Done.
Initial config parameters:              value    min   max
mu0     20.023800 -30.00  30.0
delta    9.154000   0.00  24.0
sigma0   3.919000   1.00  14.0
sigmav   0.115000   0.01   1.0
lambda  60.000000  40.00  80.0
gamma1   0.663000   0.00   1.0
gamma2   1.000000   0.00   1.0
halted   0.000002   0.00   1.0
Running 100 epochs...
Epoch: 0
Optimization [1]: Current loss: 0.005926170430423286
Optimization [101]: Current loss: 0.005792984253223923
Optimization [201]: Current loss: 0.005784885842793656
Optimization [301]: Current loss: 0.0057374942028473154
Optimization [401]: Current loss: 0.005677965897680377
Optimization [501]: Current loss: 0.0056715509477

KeyboardInterrupt: 

In [59]:
from src.config import load_default_chrom_configs
from src.optimize_H import create_bounds_params_from_config
from src.RealDataReplication import RealDataReplicationDeconvolution
from src.utils import print_fl

np.random.seed(123)
replicate = 2
chrom = 4

# Load the default replication chrom configuration from disk
# use the posterios from the CLOCCS fits to initialize
print_fl("Loading initial cell cycle parameters from CLOCCS fits.")
config1, config2 = load_default_chrom_configs(from_CLOCCS=False)
config = config1 if replicate == 1 else config2

replication_deconvolver = RealDataReplicationDeconvolution(config, 
    replicate=replicate, chr=chrom)

# Generate initial parameters and boundaries for optimization
bounds_df = create_bounds_params_from_config(config)


Loading initial cell cycle parameters from CLOCCS fits.
todo: Loading testing config from replication deconvolution
Threshold windows with less than 75% read coverage.


In [62]:
from src.optimize_H import ParameterOptimizer

num_iterations_N_B = 10

# First iteration to settle N, Fr, and B
print_fl("Running initial iterations...")
replication_deconvolver.setup_deconvolution(config)
replication_deconvolver.iterative_deconvolution_updates(num_iterations_N_B, verbose=False)
print_fl("Done.")

print("Initial config parameters: ", bounds_df)

# Parameter Optimizer
optimizer = ParameterOptimizer(
    init_params_df=bounds_df,
    config=config,
    N=replication_deconvolver.N,
    F=replication_deconvolver.F,
    B=replication_deconvolver.B,
    G=replication_deconvolver.G
)


Running initial iterations...
Initializing N using config H.
Initializing B using timepoint 0
Done.
Initial config parameters:              value    min   max
mu0     20.023800 -30.00  30.0
delta    9.154000   0.00  24.0
sigma0   3.919000   1.00  14.0
sigmav   0.115000   0.01   1.0
lambda  60.000000  40.00  80.0
gamma1   0.663000   0.00   1.0
gamma2   1.000000   0.00   1.0
halted   0.000002   0.00   1.0


In [64]:

# Parameter updates df
# print_fl(f"Running {num_epochs} epochs...")
# update_params_df, Hs, Fs, Ns, Bs = run_epochs(replication_deconvolver, optimizer, 
# num_epochs)

# Run the optimizer
#run_epochs(replication_deconvolver, optimizer, num_epochs, function_update=epoch_updates)

optimizer.optimize(maxiter=1000, verbose=True)


Optimization [1]: Current loss: 0.003713674276477069
Optimization [101]: Current loss: 0.003592480097769512
Optimization [201]: Current loss: 0.0035747893000146495
Optimization [301]: Current loss: 0.003524238904221516
Optimization [401]: Current loss: 0.0035166586547504255
Optimization [501]: Current loss: 0.0035161014551735813
Optimization [601]: Current loss: 0.003514376083494087
Optimization [701]: Current loss: 0.003498767633913598
Optimization [801]: Current loss: 0.003492305515856442
Optimization [901]: Current loss: 0.0034914219895494244
Optimization [1001]: Current loss: 0.0034914087808711235
Optimization terminated successfully.
         Current function value: 0.003491
         Iterations: 652
         Function evaluations: 1017


In [65]:
# Initialize N and B from the previous 20 iterations setup
N, B = replication_deconvolver.N, replication_deconvolver.B

In [66]:
from src.RealDataReplication import compute_rn

loss = compute_rn(replication_deconvolver.N, 
    optimizer.current_H, optimizer.F, optimizer.B, optimizer.G)
loss

0.0034914083828340927

In [69]:
from src.timer import Timer

timer = Timer()

num_iterations_N_B = 1

replication_deconvolver.H = optimizer.current_H

replication_deconvolver.test_F = optimizer.F

replication_deconvolver.iterative_deconvolution_updates(
    total_iterations=num_iterations_N_B, timer=timer,
    initial_B=B, initial_N=N, verbose=True)
timer.print_time()

# params_row = pd.DataFrame([optimizer.params_df['value']], index=[epoch])
# params_row['opt_H_loss'] = optimizer.rn
# params_row['F_rn'] = replication_deconvolver.rn

# update_params_df = pd.concat([update_params_df, params_row])

# optimizer.N = replication_deconvolver.N
# optimizer.B = replication_deconvolver.B
# optimizer.F = replication_deconvolver.F


Iteration 0
0/704 - 00:00:00.016
100/704 - 00:00:00.068
200/704 - 00:00:00.117
300/704 - 00:00:00.166
400/704 - 00:00:00.228
500/704 - 00:00:00.342
600/704 - 00:00:00.431
700/704 - 00:00:00.508
Current rn debug:  0.31249649905634375


ValueError: Testing